In [ ]:
!pip uninstall -y langchain langchain-core langchain-community langchain-openai langchain-text-splitters langchain-google-genai langsmith
!pip install -q -U langchain langchain-core langchain-google-genai

In [ ]:
from getpass import getpass
import os

GEMINI_KEY = getpass("Gemini API Key Here: ")
os.environ["GOOGLE_API_KEY"] = GEMINI_KEY


In [ ]:
import google.generativeai as genai
for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(model.name)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

gemini = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0
)

In [ ]:
reviews = [
    f"""I have been using this wireless mouse for the last 3 months and it has exceeded my expectations. The battery life is excellent and the ergonomics make it comfortable for long working hours. Connectivity is seamless and I haven't faced any lag issues. Highly recommended for professionals and students.""",

    f"""The laptop performance is quite impressive for the price. Applications load quickly and multitasking is smooth. However, the speakers could have been better and the webcam quality is average. Overall, it is still a great purchase.""",

    f"""The hotel was decent. The room was clean and check-in was smooth. The location was convenient but nothing extraordinary. Breakfast options were limited. It was an average experience overall.""",

    f"""I am disappointed with this product. The build quality feels cheap and the battery drains within a few hours. Customer support was slow to respond and did not resolve my issue satisfactorily. I would not buy this again.""",

    f"""This was one of the worst online shopping experiences I have had. The item arrived damaged, the packaging was poor, and the seller refused to provide a replacement. Complete waste of money and time."""
]

Define Output Parser

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class ReviewAnalysisResponse(BaseModel):
  summary: str = Field(description = "A brief summary of customer review with maximum 3 lines")
  positives: list = Field(description = "A list showing the positives mentioned by the customer in the review if any- max 3 points")
  negatives: list = Field(description = "A list showing the negatives mentioned by the customer in the review if any- max 3 points")
  sentiment: str = Field(description = "One word showing the sentimwnt of review - positive, negative or neutral")
  emotions: list = Field(description = "A list of 3-5 emotions expressed by the customer in review")
  email: str = Field(description = "Detailed email to customer based on the sentiment")

parser = PydanticOutputParser(pydantic_object=ReviewAnalysisResponse)


In [ ]:
prompt = PromptTemplate(
    template="""
You are a review analyst agent.

Analyze the following customer review and return the output strictly in the given JSON format.

Customer Review:
{review}

{format_instructions}
""",
    input_variables=["review"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

Create a LCEL LLM Chain

In [ ]:
chain = (prompt | gemini | parser)

Format the input reviews

In [ ]:
reviews_formatted = [{'review':review} for review in reviews]
reviews_formatted[0]

Get response from LLM

In [ ]:
responses = chain.map().invoke(reviews_formatted)

In [ ]:
responses[0]

In [ ]:
responses[0].dict()

In [ ]:
for response in responses:
  for k,v in response.dict().items():
    print(f"{k}: {v}")
  print('--------')
  print('\n')

Project 2 : Research Paper Analyst

In [ ]:
paper_abstracts = [

    f"""
    Generative Artificial Intelligence (GenAI) has emerged as a transformative technology in healthcare. This study evaluates the effectiveness of large language models in assisting medical professionals with diagnosis support and clinical documentation. A dataset of 50,000 anonymized patient records was used to compare AI-assisted and traditional workflows. Results indicate that GenAI reduced documentation time by 35% and improved diagnostic consistency by 12%. However, concerns regarding hallucinations and patient privacy remain significant challenges. Future research should focus on improving model reliability and regulatory compliance.
    """,

    f"""
    The adoption of electric vehicles (EVs) has accelerated globally due to environmental concerns and government incentives. This paper analyzes the factors influencing consumer adoption of EVs across emerging markets. Using survey data from 5,000 respondents across five countries, the study identifies charging infrastructure, vehicle cost, and driving range as the most significant determinants. The findings suggest that policy interventions and investments in charging networks can substantially increase EV adoption rates. The study contributes to the growing literature on sustainable transportation.
    """,

    f"""
    Financial fraud continues to impose substantial losses on businesses worldwide. This research proposes a machine learning framework for real-time fraud detection using transaction-level data. Various algorithms, including Random Forest, XGBoost, and Neural Networks, were evaluated on a dataset containing over one million transactions. The proposed framework achieved an accuracy of 98.2% and significantly reduced false positives compared to traditional rule-based systems. The study demonstrates the effectiveness of machine learning techniques in enhancing financial security and operational efficiency.
    """,

    f"""
    The COVID-19 pandemic accelerated the adoption of remote work arrangements across industries. This paper investigates the impact of remote work on employee productivity and job satisfaction. Data collected from 2,500 professionals across technology, finance, and consulting sectors revealed that remote work improved flexibility and work-life balance. However, challenges related to collaboration and employee engagement were also identified. The findings highlight the importance of hybrid work models in balancing organizational performance and employee well-being.
    """,

    f"""
    Climate change poses significant risks to global agricultural productivity. This study examines the impact of changing temperature and precipitation patterns on crop yields in South Asia. Using historical climate and agricultural data spanning three decades, the analysis reveals a negative correlation between rising temperatures and crop output. Adaptation strategies such as drought-resistant seeds and precision farming technologies were found to mitigate some of the adverse effects. The study provides actionable insights for policymakers and agricultural stakeholders.
    """
]

In [ ]:
SYS_PROMPT = """Act as an Artificial Intelligence Expert.
Transform the input research paper abstract given below
based on the instruction input by the user."""

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYS_PROMPT),
        ("human", "{instruction}")
    ]
)

In [ ]:
chain=(prompt | gemini )

In [ ]:
from langchain_core.messages import HumanMessage
prompt_txt = f"""
Based on following research paper abstract,
create the summary report of maximum10 lines
for a general audience

Abstract:
{paper_abstracts}"""

messages=[HumanMessage(content=prompt_txt)]
user_instruction = {"instruction":messages}
response = chain.invoke(user_instruction)
messages.append(response)
print(response.content)

In [ ]:
messages

In [ ]:
from langchain_core.messages import HumanMessage
prompt_txt = f"""
Use only the research paper abstract from earlier and create a detailed report for ha healthcare company
In the report, also include bullet points (3 max) for pros and cons of ethics in GenAI"""

messages.append([HumanMessage(content=prompt_txt)])
user_instruction = {"instruction":messages}
response = chain.invoke(user_instruction)
messages.append(response)
print(response.content)

Project 3 : Social Media Analyst

In [ ]:
fact_sheet_mobile= """ The Samsung Galaxy Z Fold4 5G in Phantom Black is a premium foldable smartphone launched by Samsung in 2022. The device features a 7.6-inch Dynamic AMOLED 2X foldable main display and a 6.2-inch cover display, both supporting a 120Hz refresh rate for smooth performance. It is powered by the Qualcomm Snapdragon 8+ Gen 1 processor and comes with 12GB RAM along with storage options of 256GB, 512GB, and 1TB. The smartphone runs on Android and supports advanced multitasking features, making it suitable for productivity-focused users.

The Galaxy Z Fold4 is equipped with a triple rear camera setup consisting of a 50MP primary camera, a 12MP ultra-wide camera, and a 10MP telephoto camera. It also includes a 10MP cover camera and a 4MP under-display camera on the main screen. The device houses a 4,400mAh battery with support for wired fast charging, wireless charging, and reverse wireless charging. Additional features include 5G connectivity, Wi-Fi 6E, Bluetooth 5.2, NFC, Samsung DeX support, S-Pen compatibility, and IPX8 water resistance.

The phone weighs approximately 263 grams and is positioned as a flagship foldable device designed for users seeking a combination of smartphone portability and tablet-like productivity. Its key strengths include a large immersive display, strong performance, multitasking capabilities, premium build quality, and advanced camera system. However, it is relatively expensive, heavier than traditional smartphones, and the display crease remains visible under certain viewing conditions. """

create prompt template for first advert

In [ ]:
prompt_txt= """
Act as a marketing manager.
Your task is to help a marketing team to create a
description for a retail website advert of a product based
on a technical fact sheet specifications for a mobile smartphone.

Write a product description

Technical specifications: {fact_sheet_mobile}

"""

chat_template = ChatPromptTemplate.from_template(prompt_txt)

In [ ]:
chain = (chat_template | gemini)

In [ ]:
response=chain.invoke({"fact_sheet_mobile":fact_sheet_mobile})
print(response.content)

In [ ]:
from IPython.display import display,Markdown
display(Markdown(response.content))

Create prompt template for second advert

In [ ]:
prompt_txt= """
Act as a marketing manager.
Your task is to help a marketing team to create a
description for a retail website advert of a product based
on a technical fact sheet specifications for a mobile smartphone.

The description should follow this format:

Product Name: <Name of the Smartphone>

Description : <Brief Overview of the feature>

Product specifications: <Table with key product feature specification>

The description should focus on most important features
a customer might look for in a phone including the foldable display screen, processing power, RAM, camera and battery life.

After the description, the table should have the
key specifications of the product. It should have 2 columns.
The first column should have 'Feature'
and the second column should have 'Specification'

and try to put excat numeric values for features if they exist.

Only put those features in the table - foldable display screen, processing power, RAM, camera and battery life.

Technical specifications: {fact_sheet_mobile}

"""

chat_template = ChatPromptTemplate.from_template(prompt_txt)

In [ ]:
chain = (chat_template | gemini)
response=chain.invoke({"fact_sheet_mobile":fact_sheet_mobile})
print(response.content)

In [ ]:
from IPython.display import display,Markdown
display(Markdown(response.content))

Third advert

In [ ]:
prompt_txt= """
Act as a marketing manager.
Your task is to help a marketing team to create a
description for a retail website advert of a product based
on a technical fact sheet specifications for a mobile smartphone.

Write a catchy product description with some emojis,
which uses atmost 60 words
and focuses on the most important things about the smartphone
which might matter to users like dsiplay and camera.

Technical specifications: {fact_sheet_mobile}

"""

chat_template = ChatPromptTemplate.from_template(prompt_txt)

In [ ]:
chain = (chat_template | gemini)
response=chain.invoke({"fact_sheet_mobile":fact_sheet_mobile})
print(response.content)

In [ ]:
from IPython.display import display,Markdown
display(Markdown(response.content))

IT Support Analyst

Define output parser

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

class ITSupportResponse(BaseModel):
  orig_msg: str = Field(description = "The original customer IT support query message")
  orig_lang: str = Field(description = "Detected language of the customer message e.g. Spanish")
  category: str = Field(description = "1-2 word descibing the category of the problem")
  trans_msg: str = Field(description = "Translated customer IT support query message in English")
  response: str = Field(description = "Response to the customer in their original language - orig_lang")
  trans_response: str = Field(description = "Response to the customer in English")

parser = JsonOutputParser(pydantic_object=ITSupportResponse)

In [ ]:
prompt_txt = """
Act as an Information Technology (IT) customer Support Agent.
For the IT support message mentioned below
use the follwoing output format when generating output response

Output Format instructions:
{format_instructions}

Customer IT support message:
{it_support_msg}

"""

prompt=PromptTemplate(
    template=prompt_txt,
    input_variables=["it_support_msg"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

In [ ]:
!pip install -q langchain langchain-core langchain-groq

In [ ]:
from langchain_groq import ChatGroq
from getpass import getpass

groq_key = getpass("Enter Groq API key: ")

llm = ChatGroq(
    api_key=groq_key,
    model="llama-3.1-8b-instant",
    temperature=0
)

In [ ]:
llm_chain = (prompt|llm|parser)

In [ ]:
it_support_msgs = [

    # Spanish
    """Hola, no puedo acceder a mi cuenta de correo electrónico corporativo. Cada vez que intento iniciar sesión, aparece un mensaje que indica que mi contraseña es incorrecta, aunque estoy seguro de que estoy utilizando la contraseña correcta.""",

    # French
    """Bonjour, mon ordinateur portable de bureau redémarre automatiquement toutes les quelques minutes. J'ai essayé de le redémarrer plusieurs fois mais le problème persiste.""",

    # German
    """Hallo, ich kann keine Verbindung zum Unternehmens-VPN herstellen. Die Anwendung zeigt ständig einen Verbindungsfehler an und ich kann nicht auf interne Systeme zugreifen.""",

    # Italian
    """Ciao, dopo l'ultimo aggiornamento del software il programma si blocca ogni volta che provo ad aprire un file Excel di grandi dimensioni.""",

    # Portuguese
    """Olá, minha impressora de rede não está funcionando. Ela aparece como offline e não consigo imprimir nenhum documento do meu computador.""",

    # Japanese
    """こんにちは。会社のTeamsアプリにログインできません。認証エラーが表示され、会議に参加することができません。""",

    # Korean
    """안녕하세요. 회사 노트북의 인터넷 연결이 계속 끊어집니다. 와이파이에 연결되어 있지만 웹사이트가 열리지 않습니다.""",

    # Chinese
    """您好，我无法访问公司的共享网络驱动器。系统提示权限被拒绝，但我昨天还能正常访问。""",

    # Hindi
    """नमस्ते, मेरे ऑफिस लैपटॉप पर आउटलुक ईमेल सिंक नहीं हो रहा है। नए ईमेल दिखाई नहीं दे रहे हैं और भेजे गए ईमेल आउटबॉक्स में फंसे हुए हैं।""",

    # English
    """Hi, my laptop is showing a blue screen error whenever I try to launch the accounting software. The issue started after installing the latest Windows update."""
]

In [ ]:
it_support_formatted = [
    {"it_support_msg": msg}
    for msg in it_support_msgs
]
it_support_formatted[0]
responses = llm_chain.map().invoke(it_support_formatted)

In [ ]:
responses[0]

In [ ]:
import pandas as pd
df=pd.DataFrame(responses)
df